# SynthID-Text

Short notebook to test synthid-text with llama3.1-8B

## Generate Text with SynthID-Text Watermark

In [22]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import LogitsProcessorList
from transformers import SynthIDTextWatermarkLogitsProcessor
from os import getenv
from transformers import BitsAndBytesConfig

# 4-Bit-Quantisierung konfigurieren
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

# 1. Modell und Tokenizer laden
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
HF_TOKEN = getenv("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=HF_TOKEN,
    clean_up_tokenization_spaces=False
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=HF_TOKEN,
    torch_dtype="auto",
    device_map="mps",
    quantization_config=quantization_config
)

# 2. SynthID LogitsProcessor konfigurieren
# Ein geheimer Schlüssel (keys) steuert die mathematische Pseudozufallsfunktion
synthid_processor = SynthIDTextWatermarkLogitsProcessor(
    keys=[1234, 5678, 9012],  # Pseudozufalls-Schlüssel
    sampling_table_size=1024,  # Größe der Hash-Tabelle für Logit-Shift
    ngram_len=5,
    sampling_table_seed=42,  # Seed für Determinismus
    context_history_size=5,  # Entspricht der Kontextlänge (ngram_len)
    device="mps"
)

# In die LogitsProcessorList von Hugging Face einreihen
logits_processor = LogitsProcessorList([synthid_processor])

# 3. Input vorbereiten
prompt = """Ich habe folgende Nachricht von einem Studierenden Erhalten:
[Hallo Michael, wäre es möglich eine Verlängerung für die Abgabe in
Data Science zu bekommen? Gruß, Florian]. Formuliere bitte eine freundliche,
jedoch ablehnende Antwort von Michael."""
messages = [{"role": "user", "content": prompt}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

# 4. Text generieren (mit eingebettetem Wasserzeichen)
output_ids = model.generate(
    **inputs,  # <--- Hier mit ** entpacken
    max_new_tokens=400,
    do_sample=True,
    temperature=0.7,
    logits_processor=logits_processor,
)

# 5. Output dekodieren
input_length = inputs["input_ids"].shape[1]
generated_text = tokenizer.decode(
    output_ids[0][input_length:], skip_special_tokens=True
)

print("-- Generierter Text (mit SynthID Wasserzeichen)--\n")
print(generated_text)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

-- Generierter Text (mit SynthID Wasserzeichen)--

Hallo Florian,

vielen Dank für deine Nachricht. Ich verstehe, dass du eine Verlängerung für die Abgabe in Data Science benötigst. Leider ist das nicht möglich, da unsere akademischen Fristen sehr eng sind und wir alle Studierenden auf die gleiche Zeitplanung zurückgreifen.

Wenn du Schwierigkeiten hast, die Aufgaben innerhalb der Frist abzugeben, kann ich dir jedoch helfen, indem ich dir einen Termin für eine Einzelbesprechung(homearbeit) gebe, um gemeinsam eine Lösung zu finden. Bitte melde dich bei mir.

Mit freundlichen Grüßen,
Michael


## Check generated Text

In [2]:
import torch
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import (
    AutoTokenizer,
    BayesianDetectorModel,
    PretrainedConfig,
    SynthIDTextWatermarkDetector,
    SynthIDTextWatermarkLogitsProcessor,
)

# 1. Class-Patch für transformers Kompatibilität
if not hasattr(BayesianDetectorModel, "all_tied_weights_keys"):
    BayesianDetectorModel.all_tied_weights_keys = []

model_hub_id = "joaogante/dummy_synthid_detector"

# 2. Config manuell laden und reparieren
config = PretrainedConfig.from_pretrained(model_hub_id)

if isinstance(config.watermarking_config, list):
    # Die Liste in das darin enthaltene Dictionary auflösen
    config.watermarking_config = config.watermarking_config[0]

# 3. Detektor-Modell mit der reparierten Config laden
detector_model = BayesianDetectorModel(config)

weights_path = hf_hub_download(repo_id=model_hub_id, filename="model.safetensors")
state_dict = load_file(weights_path)
detector_model.load_state_dict(state_dict)
detector_model.to("mps")

# 4. LogitsProcessor aus der korrigierten Config erstellen
synthid_processor = SynthIDTextWatermarkLogitsProcessor(
    **config.watermarking_config,
    device="mps",
)

# 5. SynthID-Detektor instanziieren
detector = SynthIDTextWatermarkDetector(
    detector_module=detector_model,
    logits_processor=synthid_processor,
    tokenizer=tokenizer,
)

# 6. Text prüfen
test_input = tokenizer(generated_text, return_tensors="pt").input_ids.to("mps")
outputs = detector(test_input)

print("\n--- SynthID Prüfergebnis ---")
print(outputs)


--- SynthID Prüfergebnis ---
(tensor([4.8855e-05], device='mps:0', grad_fn=<SigmoidBackward0>),)


**Achtung:** Das Ergebnis der Überprüfung ist nur ein Dummy-Ergebnis, da das verwendete Modell nicht korrekt trainiert ist. Dieses Modell wird nur verwendet, um Pipelines zu testen, kann aber nicht wirklich Wasserzeichen detektieren.

In [3]:
# Output-Tensor entpacken
probability = outputs[0].item()

print(f"Wasserzeichen-Wahrscheinlichkeit: {probability:.2%}")
# Ausgabe: Wasserzeichen-Wahrscheinlichkeit: 0.00%

if probability > 0.8:
    print("Ergebnis: Wasserzeichen vorhanden")
else:
    print("Ergebnis: Kein Wasserzeichen erkannt")

Wasserzeichen-Wahrscheinlichkeit: 0.00%
Ergebnis: Kein Wasserzeichen erkannt


## Sandbox